In [3]:
# =====================================================
# CONFIGURACIÓN INICIAL
# =====================================================

from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display



CARPETA_NOTEBOOKS = Path.cwd().resolve()


RAIZ_PROYECTO = CARPETA_NOTEBOOKS.parent


CARPETA_VISUALIZACIONES = RAIZ_PROYECTO / "visualizaciones"


CARPETA_VISUALIZACIONES.mkdir(
    parents=True,
    exist_ok=True
)





SRC = RAIZ_PROYECTO / "src"


if not SRC.exists():
    raise FileNotFoundError(
        f"No se encontró la carpeta src:\n{SRC}"
    )


if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))




In [4]:
# =====================================================
# IMPORTACIÓN DE MÓDULOS EBLET
# =====================================================

import encuesta_lite
import clasificador_individual

# Recargar módulos por si se han modificado durante la sesión
importlib.reload(encuesta_lite)
importlib.reload(clasificador_individual)


calcular_kpis_lite = encuesta_lite.calcular_kpis_lite
validar_respuestas = encuesta_lite.validar_respuestas

clasificar_individuo = clasificador_individual.clasificar_individuo



In [5]:
# =====================================================
# CARGA DEL CSV
# =====================================================

ruta_csv = (
    RAIZ_PROYECTO
    / "datasets"
    / "respuestas_reales.csv"
)

if not ruta_csv.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo:\n{ruta_csv}"
    )

df_raw = pd.read_csv(
    ruta_csv,
    encoding="utf-8-sig"
)

df_raw.columns = (
    df_raw.columns
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)



In [6]:
# =====================================================
# 1. CONFIGURACIÓN DE EBLET-LITE
# =====================================================

COLUMNA_ALIAS = "🎭 Mi alias es"

PREGUNTAS_LITE = [
    f"q{i}"
    for i in range(10, 33)
]

INDICADORES = [
    "Burnout",
    "Boreout",
    "Bienestar",
    "Intencion_cambio"
]


# Colores de perfiles
COLORES_PERFILES = {
    "🟢 Equilibrio": "#2E8B57",
    "🟡 Estable": "#D4A017",
    "🟡 Estable Neutro": "#D4A017",
    "🟠 Quemado Activo": "#E67E22",
    "🔵 Aburrido Crónico": "#3498DB",
    "🔴 Riesgo Dual": "#C0392B",
    "⚫ Desvinculado": "#34495E"
}


# Colores de indicadores
COLORES_KPIS = {
    "Burnout": "#C0392B",
    "Boreout": "#3498DB",
    "Bienestar": "#2E8B57",
    "Intención de cambio": "#8E5A2B"
}


# Colores de cultura
COLORES_CULTURA = {
    "Adhocracia": "#4C78A8",
    "Clan": "#59A14F",
    "Mercado": "#E15759",
    "Jerárquica": "#79706E"
}


# Benchmark incluido en clasificador_individual.py
BENCHMARK_MEDIAS = {
    "Burnout": 2.9,
    "Boreout": 2.6,
    "Bienestar": 3.2,
    "Intencion_cambio": 3.0
}


print("✅ Configuración de EBLET-Lite preparada")

✅ Configuración de EBLET-Lite preparada


In [7]:
# =====================================================
# 2. MAPEO GOOGLE FORMS → VARIABLES EBLET-LITE
# =====================================================

MAPEO_COLUMNAS = {
    # Burnout
    "Me siento emocionalmente agotado/a por mi trabajo.": "q10",
    "Trabajar todo el día es un verdadero esfuerzo.": "q11",
    "He desarrollado una actitud distante hacia mi trabajo.": "q12",
    "Me he vuelto menos entusiasta con mi trabajo.": "q13",

    # Boreout
    "Mi trabajo me resulta monótono y repetitivo.": "q14",
    "Tengo la sensación de que mi trabajo carece de sentido.": "q15",
    "A menudo me aburro en el trabajo.": "q16",
    "Siento que estoy infrautilizado/a en mis funciones.": "q17",

    # Bienestar
    "Me he sentido alegre y de buen humor.": "q18",
    "Me he sentido tranquilo/a y relajado/a.": "q19",
    "Me he sentido activo/a y vigoroso/a.": "q20",
    "He tenido energía para hacer las cosas del día a día.": "q21",

    # Intención de cambio
    "Estoy buscando activamente otro empleo.": "q22",
    "Es probable que cambie de empresa el próximo año.": "q23",
    "A veces pienso en dejar mi trabajo.": "q24",

    # Cultura: Adhocracia
    "En mi organización se fomenta experimentar con nuevas ideas.": "q25",
    "Se valora la creatividad y la innovación.": "q26",

    # Cultura: Clan
    "Existe buena colaboración entre compañeros.": "q27",
    "Mi responsable se preocupa por las personas.": "q28",

    # Cultura: Mercado
    "Los objetivos son prioritarios en mi organización.": "q29",
    "Existe presión por conseguir resultados.": "q30",

    # Cultura: Jerárquica
    "Los procedimientos están claramente definidos.": "q31",
    "Hay normas y reglas estrictas que seguir en el día a día": "q32"
}




In [8]:
# =====================================================
# 3. VALIDACIÓN DE COLUMNAS
# =====================================================

if COLUMNA_ALIAS not in df_raw.columns:
    posibles_alias = [
        columna
        for columna in df_raw.columns
        if "alias" in columna.lower()
    ]

    raise KeyError(
        f"No se encontró la columna:\n'{COLUMNA_ALIAS}'\n\n"
        f"Columnas que podrían corresponder al alias:\n"
        f"{posibles_alias}"
    )


columnas_forms_ausentes = [
    columna
    for columna in MAPEO_COLUMNAS
    if columna not in df_raw.columns
]


if columnas_forms_ausentes:
    print("❌ No se han encontrado estas columnas:\n")

    for columna in columnas_forms_ausentes:
        print(f" - {columna}")

    raise KeyError(
        "\nEl texto de algunas preguntas no coincide exactamente "
        "con el encabezado del CSV."
    )



In [9]:
# =====================================================
# 4. PREPARACIÓN DE LOS DATOS
# =====================================================

columnas_mapeadas = {
    columna_original: nombre_interno
    for columna_original, nombre_interno in MAPEO_COLUMNAS.items()
}


df = df_raw.rename(
    columns=columnas_mapeadas
)


df_procesar = df[
    [COLUMNA_ALIAS, *PREGUNTAS_LITE]
].copy()


# Normalizar alias
df_procesar[COLUMNA_ALIAS] = (
    df_procesar[COLUMNA_ALIAS]
    .fillna("")
    .astype(str)
    .str.strip()
)


# Crear alias para respuestas sin identificador
mascara_alias_vacio = (
    df_procesar[COLUMNA_ALIAS] == ""
)

df_procesar.loc[
    mascara_alias_vacio,
    COLUMNA_ALIAS
] = [
    f"Anónimo_{indice + 1}"
    for indice in df_procesar.index[mascara_alias_vacio]
]


# Convertir las respuestas a formato numérico
for pregunta in PREGUNTAS_LITE:
    df_procesar[pregunta] = pd.to_numeric(
        df_procesar[pregunta],
        errors="coerce"
    )




In [10]:
# =====================================================
# 5. CÁLCULO DE KPIs Y CLASIFICACIÓN INDIVIDUAL
# =====================================================

resultados = []
filas_descartadas = []


for indice, fila in df_procesar.iterrows():

    alias = fila[COLUMNA_ALIAS]

    # Extraer las 23 respuestas
    respuestas = {
        pregunta: fila[pregunta]
        for pregunta in PREGUNTAS_LITE
        if pd.notna(fila[pregunta])
    }

    # Validación inicial
    valido, mensaje = validar_respuestas(respuestas)

    if not valido:
        filas_descartadas.append({
            "Fila_CSV": indice + 2,
            "Alias": alias,
            "Motivo": mensaje
        })
        continue

    # Convertir a enteros una vez validadas
    respuestas = {
        pregunta: int(valor)
        for pregunta, valor in respuestas.items()
    }

    # KPIs
    kpis = calcular_kpis_lite(respuestas)

    # Perfil
    clasificacion = clasificar_individuo(kpis)

    percentiles = clasificacion["percentiles"]

    resultados.append({
        "Alias": alias,

        # Perfil
        "Perfil_codigo": clasificacion["perfil"],
        "Perfil": clasificacion["nombre"],
        "Descripcion_perfil": clasificacion["descripcion"],

        # KPIs
        "Burnout": round(float(kpis["burnout"]), 2),
        "Boreout": round(float(kpis["boreout"]), 2),
        "Bienestar": round(float(kpis["bienestar"]), 2),
        "Intencion_cambio": round(float(kpis["rotacion"]), 2),

        # Cultura
        "Adhocracia": round(float(kpis["cvf_adhocracia"]), 2),
        "Clan": round(float(kpis["cvf_clan"]), 2),
        "Mercado": round(float(kpis["cvf_mercado"]), 2),
        "Jerarquica": round(float(kpis["cvf_jerarquica"]), 2),
        "Cultura_dominante": kpis["cultura_dominante"],

        # Percentiles reales
        "Percentil_burnout": percentiles["burnout"],
        "Percentil_boreout": percentiles["boreout"],
        "Percentil_bienestar": percentiles["bienestar"],
        "Percentil_intencion_cambio": percentiles["rotacion"]
    })


df_resultados = pd.DataFrame(resultados)

df_descartadas = pd.DataFrame(filas_descartadas)


if df_resultados.empty:
    raise ValueError(
        "No se ha procesado ninguna respuesta válida."
    )


print("\n" + "=" * 65)
print("RESULTADO DEL PROCESAMIENTO")
print("=" * 65)

print(f"✅ Participantes procesados: {len(df_resultados)}")
print(f"⚠️ Respuestas descartadas: {len(df_descartadas)}")


if not df_descartadas.empty:
    print("\nDetalle de respuestas descartadas:")
    display(df_descartadas)


RESULTADO DEL PROCESAMIENTO
✅ Participantes procesados: 22
⚠️ Respuestas descartadas: 0


In [11]:
# =====================================================
# 6. CONTROL DE CALIDAD
# =====================================================

columnas_numericas = [
    "Burnout",
    "Boreout",
    "Bienestar",
    "Intencion_cambio",
    "Adhocracia",
    "Clan",
    "Mercado",
    "Jerarquica"
]


fuera_de_rango = (
    (df_resultados[columnas_numericas] < 1)
    | (df_resultados[columnas_numericas] > 5)
)


if fuera_de_rango.any().any():
    raise ValueError(
        "Se han detectado puntuaciones fuera del rango 1-5."
    )


columnas_percentiles = [
    "Percentil_burnout",
    "Percentil_boreout",
    "Percentil_bienestar",
    "Percentil_intencion_cambio"
]


percentiles_fuera_rango = (
    (df_resultados[columnas_percentiles] < 0)
    | (df_resultados[columnas_percentiles] > 100)
)


if percentiles_fuera_rango.any().any():
    raise ValueError(
        "Se han detectado percentiles fuera del rango 0-100."
    )


print("✅ Todos los KPIs están dentro del rango 1-5")
print("✅ Todos los percentiles están dentro del rango 0-100")
print(f"✅ Registros finales: {df_resultados.shape[0]}")

✅ Todos los KPIs están dentro del rango 1-5
✅ Todos los percentiles están dentro del rango 0-100
✅ Registros finales: 22


In [12]:
# =====================================================
# 7. RESULTADOS INDIVIDUALES
# =====================================================

tabla_individual = df_resultados[
    [
        "Alias",
        "Perfil",
        "Burnout",
        "Boreout",
        "Bienestar",
        "Intencion_cambio",
        "Cultura_dominante"
    ]
].copy()


tabla_individual = tabla_individual.rename(
    columns={
        "Intencion_cambio": "Intención de cambio",
        "Cultura_dominante": "Cultura dominante"
    }
)


display(
    tabla_individual
    .sort_values("Alias")
    .reset_index(drop=True)
    .style
    .format({
        "Burnout": "{:.2f}",
        "Boreout": "{:.2f}",
        "Bienestar": "{:.2f}",
        "Intención de cambio": "{:.2f}"
    })
)

,Alias,Perfil,Burnout,Boreout,Bienestar,Intención de cambio,Cultura dominante
0,Aft3rLife,🟠 Riesgo de Burnout,4.50,3.25,3.50,4.00,Adhocracia
1,Alex,🟡 Estable,2.25,2.75,3.50,4.33,Adhocracia
2,Bigotes,🟠 Riesgo de Burnout,4.25,2.00,2.75,4.00,Mercado
3,Estrella,🔵 Riesgo de Boreout,3.25,5.00,3.25,3.67,Mercado
4,Germán,🟢 Equilibrio,1.50,1.00,4.00,3.67,Adhocracia
5,Hastaqui,🔴 Crítico,4.50,4.25,3.00,5.00,Mercado
6,JC,🟡 Estable,2.75,2.50,4.00,3.67,Clan
7,P,🟡 Estable,2.50,2.00,4.00,4.00,Clan
8,PS,🔵 Riesgo de Boreout,2.75,3.50,3.25,4.67,Mercado
9,Phan,🟠 Riesgo de Burnout,4.50,3.25,2.25,5.00,Clan


In [22]:
# =====================================================
# CULTURA PERCIBIDA POR PARTICIPANTE
# HEATMAP DE PUNTUACIONES CVF
# =====================================================

import pandas as pd
import plotly.express as px

from estilos import aplicar_estilo


# =====================================================
# 1. COMPROBAR DATAFRAME
# =====================================================

if "df_resultados" not in globals():
    raise NameError(
        "No existe el dataframe 'df_resultados'. "
        "Ejecuta primero la celda de cálculo de KPIs."
    )


# =====================================================
# 2. DETECTAR COLUMNAS NECESARIAS
# =====================================================

columnas_obligatorias = [
    "Alias",
    "Perfil"
]

columnas_faltantes = [
    columna
    for columna in columnas_obligatorias
    if columna not in df_resultados.columns
]

if columnas_faltantes:
    raise KeyError(
        "Faltan las siguientes columnas en df_resultados: "
        + ", ".join(columnas_faltantes)
    )


# Posibles nombres utilizados para cada cultura
posibles_columnas_cultura = {
    "Adhocracia": [
        "Cultura_Adhocracia",
        "cultura_adhocracia",
        "Adhocracia"
    ],
    "Clan": [
        "Cultura_Clan",
        "cultura_clan",
        "Clan"
    ],
    "Mercado": [
        "Cultura_Mercado",
        "cultura_mercado",
        "Mercado"
    ],
    "Jerarquica": [
        "Cultura_Jerarquica",
        "Cultura_Jerárquica",
        "cultura_jerarquica",
        "cultura_jerárquica",
        "Jerarquica",
        "Jerárquica"
    ]
}


def encontrar_columna(df, nombres_posibles):
    """
    Devuelve el primer nombre de columna que exista
    en el dataframe.
    """

    for nombre in nombres_posibles:
        if nombre in df.columns:
            return nombre

    return None


columnas_detectadas = {
    cultura: encontrar_columna(
        df_resultados,
        nombres
    )
    for cultura, nombres in posibles_columnas_cultura.items()
}


print("Columnas culturales detectadas:")

for cultura, columna in columnas_detectadas.items():
    print(f"  {cultura}: {columna}")


culturas_no_encontradas = [
    cultura
    for cultura, columna in columnas_detectadas.items()
    if columna is None
]

if culturas_no_encontradas:
    raise KeyError(
        "No se han encontrado las columnas de las siguientes "
        "culturas en df_resultados: "
        + ", ".join(culturas_no_encontradas)
        + "\n\nColumnas disponibles:\n"
        + "\n".join(df_resultados.columns.astype(str))
    )


# =====================================================
# 3. DETECTAR CULTURA DOMINANTE
# =====================================================

posibles_columnas_dominante = [
    "Cultura_dominante",
    "cultura_dominante",
    "Cultura_Dominante"
]

columna_cultura_dominante = encontrar_columna(
    df_resultados,
    posibles_columnas_dominante
)


# =====================================================
# 4. CREAR DATAFRAME CULTURAL
# =====================================================

df_culturas = pd.DataFrame({
    "Alias": (
        df_resultados["Alias"]
        .astype("string")
        .str.strip()
    ),

    "Perfil": (
        df_resultados["Perfil"]
        .astype("string")
        .str.strip()
    ),

    "Adhocracia": pd.to_numeric(
        df_resultados[
            columnas_detectadas["Adhocracia"]
        ],
        errors="coerce"
    ),

    "Clan": pd.to_numeric(
        df_resultados[
            columnas_detectadas["Clan"]
        ],
        errors="coerce"
    ),

    "Mercado": pd.to_numeric(
        df_resultados[
            columnas_detectadas["Mercado"]
        ],
        errors="coerce"
    ),

    "Jerarquica": pd.to_numeric(
        df_resultados[
            columnas_detectadas["Jerarquica"]
        ],
        errors="coerce"
    )
})


if columna_cultura_dominante is not None:

    df_culturas["Cultura_dominante"] = (
        df_resultados[
            columna_cultura_dominante
        ]
        .astype("string")
        .str.strip()
    )

else:

    # Calcular cultura dominante si no existe la columna
    columnas_cvf = [
        "Adhocracia",
        "Clan",
        "Mercado",
        "Jerarquica"
    ]

    df_culturas["Cultura_dominante"] = (
        df_culturas[columnas_cvf]
        .idxmax(axis=1)
    )


# Eliminar filas sin puntuaciones culturales
df_culturas = df_culturas.dropna(
    subset=[
        "Adhocracia",
        "Clan",
        "Mercado",
        "Jerarquica"
    ],
    how="all"
).copy()


if df_culturas.empty:
    raise ValueError(
        "No existen puntuaciones culturales válidas "
        "para construir el gráfico."
    )


# =====================================================
# 5. ORDENAR PARTICIPANTES
# =====================================================

# Orden de perfiles para facilitar la lectura
ORDEN_PERFILES = [
    "🟢 Equilibrio",
    "🟡 Estable",
    "🟠 Riesgo de Burnout",
    "🔵 Riesgo de Boreout",
    "🔴 Crítico"
]


df_culturas["_orden_perfil"] = pd.Categorical(
    df_culturas["Perfil"],
    categories=ORDEN_PERFILES,
    ordered=True
)


df_culturas = (
    df_culturas
    .sort_values(
        by=[
            "_orden_perfil",
            "Alias"
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


# =====================================================
# 6. CREAR ETIQUETA DEL EJE Y
# =====================================================

df_culturas["Etiqueta"] = (
    df_culturas["Alias"]
    + " — "
    + df_culturas["Perfil"]
)


# =====================================================
# 7. MATRIZ PARA EL HEATMAP
# =====================================================

columnas_cvf = [
    "Adhocracia",
    "Clan",
    "Mercado",
    "Jerarquica"
]


matriz_cultura = df_culturas[
    columnas_cvf
]


# =====================================================
# 8. CREAR HEATMAP
# =====================================================

fig_cultura = px.imshow(
    matriz_cultura,

    x=[
        "Adhocracia",
        "Clan",
        "Mercado",
        "Jerárquica"
    ],

    y=df_culturas["Etiqueta"],

    color_continuous_scale=[
        [0.00, "#F7FBFF"],
        [0.25, "#DCEAF4"],
        [0.50, "#9CC4DD"],
        [0.75, "#4F8EB8"],
        [1.00, "#1F4E79"]
    ],

    zmin=1,
    zmax=5,

    text_auto=".1f",

    aspect="auto",

    labels={
        "x": "Tipo de cultura organizacional",
        "y": "Participante",
        "color": "Puntuación"
    },

    title="Cultura organizacional percibida por participante"
)


# =====================================================
# 9. PERSONALIZAR TOOLTIP
# =====================================================

fig_cultura.update_traces(
    textfont=dict(
        size=12
    ),

    hovertemplate=(
        "<b>%{y}</b><br>"
        "Cultura: %{x}<br>"
        "Puntuación: %{z:.2f}/5"
        "<extra></extra>"
    )
)


# =====================================================
# 10. AÑADIR CULTURA DOMINANTE AL EJE
# =====================================================

etiquetas_eje_y = [
    (
        f"{fila['Alias']} — {fila['Perfil']}"
        f"<br><span style='font-size:10px'>"
        f"Dominante: {fila['Cultura_dominante']}"
        f"</span>"
    )
    for _, fila in df_culturas.iterrows()
]


fig_cultura.update_yaxes(
    tickmode="array",
    tickvals=df_culturas["Etiqueta"],
    ticktext=etiquetas_eje_y,
    automargin=True
)


# =====================================================
# 11. TAMAÑO DINÁMICO
# =====================================================

numero_participantes = len(df_culturas)

alto_grafico = max(
    650,
    65 * numero_participantes + 220
)


fig_cultura = aplicar_estilo(
    fig_cultura,
    titulo="Cultura organizacional percibida por participante",
    ancho=1100,
    alto=alto_grafico
)


fig_cultura.update_layout(
    margin=dict(
        l=280,
        r=90,
        t=110,
        b=90
    ),

    coloraxis_colorbar=dict(
        title="Puntuación",
        tickvals=[
            1,
            2,
            3,
            4,
            5
        ],
        ticktext=[
            "1",
            "2",
            "3",
            "4",
            "5"
        ],
        len=0.75
    )
)


fig_cultura.update_xaxes(
    side="top",
    tickfont=dict(
        size=13
    ),
    title_text=None
)


fig_cultura.update_yaxes(
    title_text=None,
    tickfont=dict(
        size=11
    )
)


fig_cultura.show()


# =====================================================
# 12. DISTRIBUCIÓN DE CULTURAS DOMINANTES
# =====================================================

distribucion_culturas = (
    df_culturas["Cultura_dominante"]
    .value_counts()
    .rename_axis("Cultura dominante")
    .reset_index(name="Participantes")
)


distribucion_culturas["Porcentaje"] = (
    distribucion_culturas["Participantes"]
    / len(df_culturas)
    * 100
).round(1)


orden_culturas = [
    "Adhocracia",
    "Clan",
    "Mercado",
    "Jerarquica",
    "Jerárquica"
]


distribucion_culturas["_orden"] = (
    distribucion_culturas[
        "Cultura dominante"
    ]
    .map({
        cultura: indice
        for indice, cultura in enumerate(
            orden_culturas
        )
    })
)


distribucion_culturas = (
    distribucion_culturas
    .sort_values(
        by=[
            "_orden",
            "Participantes"
        ],
        ascending=[
            True,
            False
        ],
        na_position="last"
    )
    .drop(
        columns="_orden"
    )
    .reset_index(drop=True)
)


print(
    "\n📊 Distribución de culturas dominantes:"
)


display(
    distribucion_culturas[
        [
            "Cultura dominante",
            "Participantes",
            "Porcentaje"
        ]
    ]
)


# =====================================================
# 13. TABLA DETALLADA
# =====================================================

tabla_cultura_participantes = df_culturas[
    [
        "Alias",
        "Perfil",
        "Cultura_dominante",
        "Adhocracia",
        "Clan",
        "Mercado",
        "Jerarquica"
    ]
].copy()


tabla_cultura_participantes = (
    tabla_cultura_participantes
    .rename(
        columns={
            "Cultura_dominante": "Cultura dominante",
            "Jerarquica": "Jerárquica"
        }
    )
)


display(
    tabla_cultura_participantes
)

Columnas culturales detectadas:
  Adhocracia: Adhocracia
  Clan: Clan
  Mercado: Mercado
  Jerarquica: Jerarquica



📊 Distribución de culturas dominantes:


,Cultura dominante,Participantes,Porcentaje
0,Adhocracia,6,27.3
1,Clan,4,18.2
2,Mercado,12,54.5


,Alias,Perfil,Cultura dominante,Adhocracia,Clan,Mercado,Jerárquica
0,Germán,🟢 Equilibrio,Adhocracia,4.5,4.0,2.5,3.0
1,Alex,🟡 Estable,Adhocracia,5.0,3.5,3.5,4.0
2,JC,🟡 Estable,Clan,1.5,4.5,2.0,2.5
3,P,🟡 Estable,Clan,2.0,3.0,2.0,2.5
4,meli,🟡 Estable,Mercado,2.5,3.0,5.0,4.5
5,zyx_zyx_zyx,🟡 Estable,Mercado,3.5,3.5,4.0,3.0
6,Aft3rLife,🟠 Riesgo de Burnout,Adhocracia,4.5,1.5,1.5,1.5
7,Bigotes,🟠 Riesgo de Burnout,Mercado,3.0,2.5,5.0,3.0
8,Phan,🟠 Riesgo de Burnout,Clan,3.0,4.0,4.0,3.5
9,Wombat,🟠 Riesgo de Burnout,Adhocracia,4.0,3.5,2.5,1.5


In [14]:
# =====================================================
# MAPA DE POSICIONAMIENTO INDIVIDUAL
# BURNOUT VS. BOREOUT
# ESTILO LEGACYTECH
# =====================================================

import numpy as np
import pandas as pd
import plotly.express as px


# -----------------------------------------------------
# Umbrales
# -----------------------------------------------------

UMBRAL_BURNOUT = 3.5
UMBRAL_BOREOUT = 3.5


# -----------------------------------------------------
# Colores utilizados en LegacyTech
# Solo cuatro zonas
# -----------------------------------------------------

COLORES_ZONAS = {
    "Equilibrio / Estable": "#5B8A72",
    "Riesgo de Burnout": "#B36A3C",
    "Riesgo de Boreout": "#3E6C8F",
    "Crítico": "#8F2D2D"
}


# Orden fijo de la leyenda
ORDEN_ZONAS = [
    "Equilibrio / Estable",
    "Riesgo de Burnout",
    "Riesgo de Boreout",
    "Crítico"
]




df_mapa = df_resultados.copy()


# -----------------------------------------------------
# Clasificación de la zona del mapa
# Esta clasificación utiliza exactamente el umbral 3.5
# Los valores iguales a 3.5 entran en riesgo
# -----------------------------------------------------

def clasificar_zona_mapa(row):
    burnout = row["Burnout"]
    boreout = row["Boreout"]

    if (
        burnout >= UMBRAL_BURNOUT
        and boreout >= UMBRAL_BOREOUT
    ):
        return "Crítico"

    elif burnout >= UMBRAL_BURNOUT:
        return "Riesgo de Burnout"

    elif boreout >= UMBRAL_BOREOUT:
        return "Riesgo de Boreout"

    else:
        return "Equilibrio / Estable"


df_mapa["Zona_mapa"] = df_mapa.apply(
    clasificar_zona_mapa,
    axis=1
)


# Convertir a categoría para conservar el orden
df_mapa["Zona_mapa"] = pd.Categorical(
    df_mapa["Zona_mapa"],
    categories=ORDEN_ZONAS,
    ordered=True
)


# -----------------------------------------------------
# Separación visual de puntos superpuestos
#
# IMPORTANTE:
# No se modifican los valores reales.
# Solo se crean coordenadas auxiliares para el gráfico.
# -----------------------------------------------------

df_mapa["Burnout_plot"] = df_mapa["Burnout"].astype(float)
df_mapa["Boreout_plot"] = df_mapa["Boreout"].astype(float)


# Número de orden dentro de cada grupo coincidente
df_mapa["_orden_superposicion"] = (
    df_mapa
    .groupby(
        ["Burnout", "Boreout"],
        observed=True
    )
    .cumcount()
)


# Número total de personas en la misma posición
df_mapa["_total_superpuestos"] = (
    df_mapa
    .groupby(
        ["Burnout", "Boreout"],
        observed=True
    )["Alias"]
    .transform("size")
)


# -----------------------------------------------------
# Patrones de desplazamiento
#
# Los puntos coincidentes se distribuyen alrededor
# de su posición real.
# -----------------------------------------------------

PATRONES_DESPLAZAMIENTO = [
    (0.00, 0.00),
    (-0.08, 0.00),
    (0.08, 0.00),
    (0.00, -0.08),
    (0.00, 0.08),
    (-0.06, -0.06),
    (0.06, 0.06),
    (-0.06, 0.06),
    (0.06, -0.06),
    (-0.12, 0.00),
    (0.12, 0.00),
    (0.00, -0.12),
    (0.00, 0.12)
]


def obtener_desplazamiento(orden, total):
    """
    Devuelve un desplazamiento visual.

    Si solo existe un punto en una posición,
    no se aplica desplazamiento.
    """

    if total <= 1:
        return 0.0, 0.0

    indice = int(orden) % len(PATRONES_DESPLAZAMIENTO)

    return PATRONES_DESPLAZAMIENTO[indice]


desplazamientos = df_mapa.apply(
    lambda row: obtener_desplazamiento(
        row["_orden_superposicion"],
        row["_total_superpuestos"]
    ),
    axis=1
)


df_mapa["_desplazamiento_x"] = [
    valor[0]
    for valor in desplazamientos
]

df_mapa["_desplazamiento_y"] = [
    valor[1]
    for valor in desplazamientos
]


df_mapa["Burnout_plot"] = (
    df_mapa["Burnout"]
    + df_mapa["_desplazamiento_x"]
)

df_mapa["Boreout_plot"] = (
    df_mapa["Boreout"]
    + df_mapa["_desplazamiento_y"]
)


# -----------------------------------------------------
# Posición alterna de los alias
#
# Evita que todos los nombres aparezcan encima del punto.
# -----------------------------------------------------

POSICIONES_TEXTO = [
    "top center",
    "bottom center",
    "middle right",
    "middle left",
    "top right",
    "top left",
    "bottom right",
    "bottom left"
]


df_mapa["_posicion_texto"] = [
    POSICIONES_TEXTO[
        indice % len(POSICIONES_TEXTO)
    ]
    for indice in range(len(df_mapa))
]


# -----------------------------------------------------
# Crear el gráfico
# -----------------------------------------------------

fig_posicionamiento = px.scatter(
    df_mapa,
    x="Burnout_plot",
    y="Boreout_plot",
    text="Alias",
    color="Zona_mapa",
    category_orders={
        "Zona_mapa": ORDEN_ZONAS
    },
    color_discrete_map=COLORES_ZONAS,
    hover_name="Alias",
    custom_data=[
        "Burnout",
        "Boreout",
        "Bienestar",
        "Intencion_cambio",
        "Cultura_dominante",
        "Perfil",
        "Zona_mapa"
    ],
    title=(
        "Mapa de posicionamiento individual: "
        "Burnout vs. Boreout"
    ),
    labels={
        "Burnout_plot": "Nivel de burnout",
        "Boreout_plot": "Nivel de boreout",
        "Zona_mapa": "Zona"
    }
)


# -----------------------------------------------------
# Estilo individual de cada punto
#
# Plotly crea una traza por zona.
# Se asigna una posición de texto a cada punto.
# -----------------------------------------------------

for trace in fig_posicionamiento.data:

    indices_traza = df_mapa.index[
        df_mapa["Zona_mapa"].astype(str)
        == trace.name
    ].tolist()

    posiciones_traza = (
        df_mapa.loc[
            indices_traza,
            "_posicion_texto"
        ]
        .tolist()
    )

    trace.update(
        marker=dict(
            size=18,
            opacity=0.92,
            line=dict(
                width=2,
                color="white"
            )
        ),
        textposition=posiciones_traza,
        textfont=dict(
            size=12,
            color="#263645"
        ),
        cliponaxis=False,
        hovertemplate=(
            "<b>%{hovertext}</b><br><br>"
            "Zona: %{customdata[6]}<br>"
            "Perfil: %{customdata[5]}<br>"
            "Burnout: %{customdata[0]:.2f}<br>"
            "Boreout: %{customdata[1]:.2f}<br>"
            "Bienestar: %{customdata[2]:.2f}<br>"
            "Intención de cambio: %{customdata[3]:.2f}<br>"
            "Cultura dominante: %{customdata[4]}"
            "<extra></extra>"
        )
    )


# -----------------------------------------------------
# Fondos de los cuadrantes
#
# Se extienden ligeramente más allá de 1 y 5
# para que los puntos extremos no queden cortados.
# -----------------------------------------------------

LIMITE_MIN = 0.75
LIMITE_MAX = 5.30


# Equilibrio / Estable
fig_posicionamiento.add_shape(
    type="rect",
    x0=LIMITE_MIN,
    x1=UMBRAL_BURNOUT,
    y0=LIMITE_MIN,
    y1=UMBRAL_BOREOUT,
    fillcolor="rgba(91, 138, 114, 0.10)",
    line_width=0,
    layer="below"
)


# Riesgo de Burnout
fig_posicionamiento.add_shape(
    type="rect",
    x0=UMBRAL_BURNOUT,
    x1=LIMITE_MAX,
    y0=LIMITE_MIN,
    y1=UMBRAL_BOREOUT,
    fillcolor="rgba(179, 106, 60, 0.10)",
    line_width=0,
    layer="below"
)


# Riesgo de Boreout
fig_posicionamiento.add_shape(
    type="rect",
    x0=LIMITE_MIN,
    x1=UMBRAL_BURNOUT,
    y0=UMBRAL_BOREOUT,
    y1=LIMITE_MAX,
    fillcolor="rgba(62, 108, 143, 0.10)",
    line_width=0,
    layer="below"
)


# Crítico
fig_posicionamiento.add_shape(
    type="rect",
    x0=UMBRAL_BURNOUT,
    x1=LIMITE_MAX,
    y0=UMBRAL_BOREOUT,
    y1=LIMITE_MAX,
    fillcolor="rgba(143, 45, 45, 0.10)",
    line_width=0,
    layer="below"
)


# -----------------------------------------------------
# Líneas de corte
# -----------------------------------------------------

fig_posicionamiento.add_vline(
    x=UMBRAL_BURNOUT,
    line_width=2,
    line_dash="dash",
    line_color="#6C757D"
)

fig_posicionamiento.add_hline(
    y=UMBRAL_BOREOUT,
    line_width=2,
    line_dash="dash",
    line_color="#6C757D"
)


# -----------------------------------------------------
# Etiquetas de los cuadrantes
#
# Se colocan cerca de los bordes, con fondo blanco,
# para evitar mezclarlas con los alias.
# -----------------------------------------------------

fig_posicionamiento.add_annotation(
    x=1.00,
    y=0.90,
    text="<b>Equilibrio / Estable</b>",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(
        size=15,
        color=COLORES_ZONAS["Equilibrio / Estable"]
    ),
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor=COLORES_ZONAS["Equilibrio / Estable"],
    borderwidth=1,
    borderpad=5
)


fig_posicionamiento.add_annotation(
    x=5.05,
    y=0.90,
    text="<b>Riesgo de Burnout</b>",
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    font=dict(
        size=15,
        color=COLORES_ZONAS["Riesgo de Burnout"]
    ),
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor=COLORES_ZONAS["Riesgo de Burnout"],
    borderwidth=1,
    borderpad=5
)


fig_posicionamiento.add_annotation(
    x=1.00,
    y=5.15,
    text="<b>Riesgo de Boreout</b>",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font=dict(
        size=15,
        color=COLORES_ZONAS["Riesgo de Boreout"]
    ),
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor=COLORES_ZONAS["Riesgo de Boreout"],
    borderwidth=1,
    borderpad=5
)


fig_posicionamiento.add_annotation(
    x=5.05,
    y=5.15,
    text="<b>Crítico</b>",
    showarrow=False,
    xanchor="right",
    yanchor="top",
    font=dict(
        size=15,
        color=COLORES_ZONAS["Crítico"]
    ),
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor=COLORES_ZONAS["Crítico"],
    borderwidth=1,
    borderpad=5
)


# -----------------------------------------------------
# Nota metodológica sobre el umbral
# -----------------------------------------------------

fig_posicionamiento.add_annotation(
    x=0.5,
    y=-0.13,
    xref="paper",
    yref="paper",
    text=(
        "Nota: las puntuaciones iguales o superiores "
        "a 3,5 se incluyen en la zona de riesgo. "
        "Los desplazamientos de puntos son únicamente visuales."
    ),
    showarrow=False,
    xanchor="center",
    font=dict(
        size=12,
        color="#5F6B73"
    )
)


# -----------------------------------------------------
# Estilo general
# -----------------------------------------------------

fig_posicionamiento.update_layout(
    template="simple_white",

    # Gráfico más grande
    width=1250,
    height=900,

    title=dict(
        x=0.02,
        xanchor="left",
        font=dict(
            size=24,
            color="#263645"
        )
    ),

    font=dict(
        family="Arial",
        size=13,
        color="#263645"
    ),

    legend=dict(
        title="",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(
            size=13
        )
    ),

    margin=dict(
        l=90,
        r=90,
        t=140,
        b=130
    ),

    paper_bgcolor="white",
    plot_bgcolor="white"
)


# -----------------------------------------------------
# Ejes
#
# El rango se amplía para que los puntos con valor 1 o 5,
# sus bordes y sus alias aparezcan completos.
# -----------------------------------------------------

fig_posicionamiento.update_xaxes(
    range=[LIMITE_MIN, LIMITE_MAX],
    tickmode="array",
    tickvals=[
        1,
        1.5,
        2,
        2.5,
        3,
        3.5,
        4,
        4.5,
        5
    ],
    ticktext=[
        "1",
        "1,5",
        "2",
        "2,5",
        "3",
        "3,5",
        "4",
        "4,5",
        "5"
    ],
    showgrid=True,
    gridcolor="#E5E9EC",
    zeroline=False,
    linecolor="#AEB6BC",
    title_font=dict(
        size=16
    )
)


fig_posicionamiento.update_yaxes(
    range=[LIMITE_MIN, LIMITE_MAX],
    tickmode="array",
    tickvals=[
        1,
        1.5,
        2,
        2.5,
        3,
        3.5,
        4,
        4.5,
        5
    ],
    ticktext=[
        "1",
        "1,5",
        "2",
        "2,5",
        "3",
        "3,5",
        "4",
        "4,5",
        "5"
    ],
    showgrid=True,
    gridcolor="#E5E9EC",
    zeroline=False,
    linecolor="#AEB6BC",
    title_font=dict(
        size=16
    )
)

# =====================================================
# GUARDAR GRÁFICO
# =====================================================



ruta_grafico_posicionamiento = (
    CARPETA_VISUALIZACIONES
    / "07_mapa_posicionamiento_individual.png"
)

fig_posicionamiento.write_image(
    ruta_grafico_posicionamiento,
    width=1250,
    height=900,
    scale=3
)



fig_posicionamiento.show()

In [15]:
# =====================================================
# DISTRIBUCIÓN DE PERFILES DE BIENESTAR
# =====================================================

import plotly.express as px

# Orden fijo de perfiles
ORDEN_PERFILES = [
    "🟢 Equilibrio",
    "🟡 Estable",
    "🟠 Riesgo de Burnout",
    "🔵 Riesgo de Boreout",
    "🔴 Crítico"
]

COLORES_PERFILES = {
    "🟢 Equilibrio": "#5B8A72",
    "🟡 Estable": "#D4B15A",
    "🟠 Riesgo de Burnout": "#C57B39",
    "🔵 Riesgo de Boreout": "#4E79A7",
    "🔴 Crítico": "#B13A3A"
}


# -----------------------------------------------------
# Distribución
# -----------------------------------------------------

distribucion = (
    df_resultados["Perfil"]
    .value_counts()
    .rename_axis("Perfil")
    .reset_index(name="Participantes")
)

# Mantener siempre el mismo orden
distribucion["Perfil"] = pd.Categorical(
    distribucion["Perfil"],
    categories=ORDEN_PERFILES,
    ordered=True
)

distribucion = (
    distribucion
    .sort_values("Perfil")
)

# Porcentaje
distribucion["Porcentaje"] = (
    distribucion["Participantes"]
    / len(df_resultados)
    * 100
).round(1)


# Texto mostrado al final de la barra
distribucion["Etiqueta"] = (
    distribucion["Participantes"].astype(str)
    + " ("
    + distribucion["Porcentaje"].astype(str)
    + "%)"
)


# -----------------------------------------------------
# Gráfico
# -----------------------------------------------------

fig_perfiles= px.bar(
    distribucion,
    x="Participantes",
    y="Perfil",
    orientation="h",
    color="Perfil",
    color_discrete_map=COLORES_PERFILES,
    text="Etiqueta"
)

fig_perfiles.update_traces(
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Participantes: %{x}<br>"
        "Porcentaje: %{customdata[0]:.1f}%"
        "<extra></extra>"
    ),
    customdata=distribucion[["Porcentaje"]]
)

fig_perfiles.update_layout(

    template="simple_white",

    title="Distribución de perfiles de bienestar",

    showlegend=False,

    width=950,
    height=500,

    xaxis_title="Número de participantes",

    yaxis_title="",

    xaxis=dict(
        dtick=1,
        range=[
            0,
            distribucion["Participantes"].max()*1.35
        ]
    ),

    font=dict(
        size=13
    ),

    margin=dict(
        l=70,
        r=70,
        t=80,
        b=60
    )
)

# -----------------------------------------------------
# Guardar gráfico
# -----------------------------------------------------

ruta_grafico_perfiles = (
    CARPETA_VISUALIZACIONES
    / "04_distribucion_perfiles_bienestar.png"
)

fig_perfiles.write_image(
    ruta_grafico_perfiles,
    width=950,
    height=500,
    scale=3
)


fig_perfiles.show()



# =====================================================
# TABLA RESUMEN
# =====================================================

display(
    distribucion[
        [
            "Perfil",
            "Participantes",
            "Porcentaje"
        ]
    ]
)

,Perfil,Participantes,Porcentaje
4,🟢 Equilibrio,1,4.5
2,🟡 Estable,5,22.7
1,🟠 Riesgo de Burnout,6,27.3
3,🔵 Riesgo de Boreout,4,18.2
0,🔴 Crítico,6,27.3


In [16]:
# =====================================================
# DISTRIBUCIÓN DE CULTURAS ORGANIZACIONALES PERCIBIDAS
# =====================================================

import pandas as pd
import plotly.express as px


# =====================================================
# 1. COMPROBAR DATAFRAME
# =====================================================

if "df_resultados" not in globals():
    raise NameError(
        "No existe el dataframe 'df_resultados'. "
        "Ejecuta primero la celda de cálculo de resultados."
    )


# =====================================================
# 2. CONFIGURACIÓN DE CULTURAS
# =====================================================

ORDEN_CULTURAS = [
    "🔵 Adhocracia",
    "🟢 Clan",
    "🟠 Mercado",
    "🟡 Jerárquica"
]


COLORES_CULTURAS = {
    "🔵 Adhocracia": "#4E79A7",
    "🟢 Clan": "#5B8A72",
    "🟠 Mercado": "#C57B39",
    "🟡 Jerárquica": "#D4B15A"
}


# =====================================================
# 3. FUNCIONES AUXILIARES
# =====================================================

def encontrar_columna(
    dataframe,
    nombres_posibles
):
    """
    Devuelve la primera columna encontrada dentro
    de una lista de nombres posibles.
    """

    for nombre in nombres_posibles:

        if nombre in dataframe.columns:
            return nombre

    return None


def normalizar_cultura(valor):
    """
    Normaliza los diferentes nombres que puede tener
    una cultura organizacional.
    """

    if pd.isna(valor):
        return pd.NA

    texto = (
        str(valor)
        .strip()
        .lower()
    )

    equivalencias = {
        "adhocracia": "🔵 Adhocracia",
        "cultura adhocrática": "🔵 Adhocracia",
        "adhocratica": "🔵 Adhocracia",
        "adhocrática": "🔵 Adhocracia",

        "clan": "🟢 Clan",
        "cultura clan": "🟢 Clan",

        "mercado": "🟠 Mercado",
        "cultura de mercado": "🟠 Mercado",

        "jerarquica": "🟡 Jerárquica",
        "jerárquica": "🟡 Jerárquica",
        "jerarquia": "🟡 Jerárquica",
        "jerarquía": "🟡 Jerárquica",
        "cultura jerarquica": "🟡 Jerárquica",
        "cultura jerárquica": "🟡 Jerárquica"
    }

    if texto in equivalencias:
        return equivalencias[texto]

    # Detección parcial por si la columna contiene texto adicional
    if "adhocr" in texto:
        return "🔵 Adhocracia"

    if "clan" in texto:
        return "🟢 Clan"

    if "mercado" in texto:
        return "🟠 Mercado"

    if "jerar" in texto:
        return "🟡 Jerárquica"

    return str(valor).strip()


# =====================================================
# 4. DETECTAR CULTURA DOMINANTE
# =====================================================

columna_cultura_dominante = encontrar_columna(
    df_resultados,
    [
        "Cultura_dominante",
        "Cultura_Dominante",
        "cultura_dominante",
        "Cultura dominante",
        "cultura dominante"
    ]
)


# =====================================================
# 5. CREAR CULTURA DOMINANTE SI NO EXISTE
# =====================================================

if columna_cultura_dominante is not None:

    df_cultura_distribucion = pd.DataFrame({
        "Cultura": df_resultados[
            columna_cultura_dominante
        ]
    })

else:

    # -------------------------------------------------
    # Detectar columnas de puntuación cultural
    # -------------------------------------------------

    columnas_cultura = {
        "Adhocracia": encontrar_columna(
            df_resultados,
            [
                "Adhocracia",
                "Cultura_Adhocracia",
                "cultura_adhocracia",
                "Score_Adhocracia",
                "score_adhocracia"
            ]
        ),

        "Clan": encontrar_columna(
            df_resultados,
            [
                "Clan",
                "Cultura_Clan",
                "cultura_clan",
                "Score_Clan",
                "score_clan"
            ]
        ),

        "Mercado": encontrar_columna(
            df_resultados,
            [
                "Mercado",
                "Cultura_Mercado",
                "cultura_mercado",
                "Score_Mercado",
                "score_mercado"
            ]
        ),

        "Jerárquica": encontrar_columna(
            df_resultados,
            [
                "Jerarquica",
                "Jerárquica",
                "Cultura_Jerarquica",
                "Cultura_Jerárquica",
                "cultura_jerarquica",
                "cultura_jerárquica",
                "Score_Jerarquica",
                "Score_Jerárquica"
            ]
        )
    }


    culturas_no_encontradas = [
        cultura
        for cultura, columna in columnas_cultura.items()
        if columna is None
    ]


    if culturas_no_encontradas:

        raise KeyError(
            "No existe una columna de cultura dominante y "
            "tampoco se han encontrado todas las puntuaciones "
            "culturales necesarias.\n\n"
            "Culturas no encontradas: "
            + ", ".join(culturas_no_encontradas)
            + "\n\nColumnas disponibles:\n"
            + "\n".join(
                df_resultados.columns.astype(str)
            )
        )


    # -------------------------------------------------
    # Crear dataframe con puntuaciones culturales
    # -------------------------------------------------

    df_scores_cultura = pd.DataFrame({
        "Adhocracia": pd.to_numeric(
            df_resultados[
                columnas_cultura["Adhocracia"]
            ],
            errors="coerce"
        ),

        "Clan": pd.to_numeric(
            df_resultados[
                columnas_cultura["Clan"]
            ],
            errors="coerce"
        ),

        "Mercado": pd.to_numeric(
            df_resultados[
                columnas_cultura["Mercado"]
            ],
            errors="coerce"
        ),

        "Jerárquica": pd.to_numeric(
            df_resultados[
                columnas_cultura["Jerárquica"]
            ],
            errors="coerce"
        )
    })


    # Cultura con puntuación más alta por participante
    cultura_calculada = (
        df_scores_cultura
        .idxmax(
            axis=1,
            skipna=True
        )
    )


    # Las filas sin ninguna puntuación permanecen vacías
    filas_sin_puntuacion = (
        df_scores_cultura
        .isna()
        .all(axis=1)
    )

    cultura_calculada.loc[
        filas_sin_puntuacion
    ] = pd.NA


    df_cultura_distribucion = pd.DataFrame({
        "Cultura": cultura_calculada
    })


# =====================================================
# 6. NORMALIZAR NOMBRES
# =====================================================

df_cultura_distribucion["Cultura"] = (
    df_cultura_distribucion["Cultura"]
    .apply(normalizar_cultura)
)


# Eliminar participantes sin resultado cultural
df_cultura_distribucion = (
    df_cultura_distribucion
    .dropna(
        subset=["Cultura"]
    )
    .copy()
)


if df_cultura_distribucion.empty:
    raise ValueError(
        "No existen datos válidos de cultura organizacional "
        "para construir el gráfico."
    )


# =====================================================
# 7. DISTRIBUCIÓN
# =====================================================

distribucion_cultura = (
    df_cultura_distribucion["Cultura"]
    .value_counts()
    .rename_axis("Cultura")
    .reset_index(name="Participantes")
)


# Incluir las cuatro culturas aunque alguna tenga cero casos
distribucion_cultura = (
    pd.DataFrame({
        "Cultura": ORDEN_CULTURAS
    })
    .merge(
        distribucion_cultura,
        on="Cultura",
        how="left"
    )
)


distribucion_cultura["Participantes"] = (
    distribucion_cultura["Participantes"]
    .fillna(0)
    .astype(int)
)


# Mantener siempre el mismo orden
distribucion_cultura["Cultura"] = pd.Categorical(
    distribucion_cultura["Cultura"],
    categories=ORDEN_CULTURAS,
    ordered=True
)


distribucion_cultura = (
    distribucion_cultura
    .sort_values("Cultura")
    .reset_index(drop=True)
)


# =====================================================
# 8. PORCENTAJES Y ETIQUETAS
# =====================================================

total_participantes_cultura = (
    distribucion_cultura[
        "Participantes"
    ]
    .sum()
)


distribucion_cultura["Porcentaje"] = (
    distribucion_cultura["Participantes"]
    / total_participantes_cultura
    * 100
).round(1)


distribucion_cultura["Etiqueta"] = (
    distribucion_cultura[
        "Participantes"
    ].astype(str)
    + " ("
    + distribucion_cultura[
        "Porcentaje"
    ].astype(str)
    + "%)"
)


# =====================================================
# 9. GRÁFICO
# =====================================================

fig_cultura = px.bar(
    distribucion_cultura,

    x="Participantes",

    y="Cultura",

    orientation="h",

    color="Cultura",

    color_discrete_map=COLORES_CULTURAS,

    category_orders={
        "Cultura": ORDEN_CULTURAS
    },

    text="Etiqueta"
)


fig_cultura.update_traces(
    textposition="outside",

    cliponaxis=False,

    hovertemplate=(
        "<b>%{y}</b><br>"
        "Participantes: %{x}<br>"
        "Porcentaje: %{customdata[0]:.1f}%"
        "<extra></extra>"
    ),

    customdata=distribucion_cultura[
        ["Porcentaje"]
    ]
)


# Evitar errores cuando todos los valores sean cero
maximo_participantes = max(
    1,
    distribucion_cultura[
        "Participantes"
    ].max()
)


fig_cultura.update_layout(
    template="simple_white",

    title=(
        "Distribución de culturas "
        "organizacionales percibidas"
    ),

    showlegend=False,

    width=950,

    height=480,

    xaxis_title="Número de participantes",

    yaxis_title="",

    xaxis=dict(
        dtick=1,

        range=[
            0,
            maximo_participantes * 1.35
        ],

        showgrid=True,

        gridcolor="#ECEFF1",

        zeroline=False
    ),

    yaxis=dict(
        categoryorder="array",

        categoryarray=ORDEN_CULTURAS,

        autorange="reversed"
    ),

    font=dict(
        size=13
    ),

    margin=dict(
        l=100,
        r=90,
        t=90,
        b=60
    )
)


# =====================================================
# GUARDAR GRÁFICO
# =====================================================

ruta_grafico_cultura = (
    CARPETA_VISUALIZACIONES
    / "05_distribucion_culturas_organizacionales.png"
)

fig_cultura.write_image(
    ruta_grafico_cultura,
    width=950,
    height=480,
    scale=3
)


fig_cultura.show()


# =====================================================
# 10. TABLA RESUMEN
# =====================================================

display(
    distribucion_cultura[
        [
            "Cultura",
            "Participantes",
            "Porcentaje"
        ]
    ]
)

,Cultura,Participantes,Porcentaje
0,🔵 Adhocracia,6,27.3
1,🟢 Clan,4,18.2
2,🟠 Mercado,12,54.5
3,🟡 Jerárquica,0,0.0


In [17]:
# =====================================================
# RELACIÓN ENTRE CULTURA ORGANIZACIONAL Y KPIs
# =====================================================

import pandas as pd
import plotly.express as px


# =====================================================
# 1. COMPROBAR DATAFRAME
# =====================================================

if "df_resultados" not in globals():
    raise NameError(
        "No existe el dataframe 'df_resultados'. "
        "Ejecuta primero la celda de cálculo de resultados."
    )


# =====================================================
# 2. CONFIGURACIÓN
# =====================================================

ORDEN_CULTURAS = [
    "🔵 Adhocracia",
    "🟢 Clan",
    "🟠 Mercado",
    "🟡 Jerárquica"
]

ORDEN_KPIS = [
    "Burnout",
    "Boreout",
    "Bienestar",
    "Intención de cambio"
]

COLORES_KPIS = {
    "Burnout": "#C57B39",
    "Boreout": "#4E79A7",
    "Bienestar": "#5B8A72",
    "Intención de cambio": "#B13A3A"
}


# =====================================================
# 3. FUNCIONES AUXILIARES
# =====================================================

def encontrar_columna(dataframe, nombres_posibles):
    """
    Devuelve la primera columna encontrada dentro
    de una lista de nombres posibles.
    """

    for nombre in nombres_posibles:
        if nombre in dataframe.columns:
            return nombre

    return None


def normalizar_cultura(valor):
    """
    Normaliza los nombres de las culturas organizacionales.
    """

    if pd.isna(valor):
        return pd.NA

    texto = str(valor).strip().lower()

    if "adhocr" in texto:
        return "🔵 Adhocracia"

    if "clan" in texto:
        return "🟢 Clan"

    if "mercado" in texto:
        return "🟠 Mercado"

    if "jerar" in texto:
        return "🟡 Jerárquica"

    return pd.NA


# =====================================================
# 4. DETECTAR COLUMNAS
# =====================================================

columna_cultura = encontrar_columna(
    df_resultados,
    [
        "Cultura_dominante",
        "Cultura_Dominante",
        "cultura_dominante",
        "Cultura dominante",
        "cultura dominante",
        "cultura_percibida",
        "Cultura_percibida"
    ]
)


columnas_kpis = {
    "Burnout": encontrar_columna(
        df_resultados,
        [
            "kpi_burnout",
            "KPI_Burnout",
            "Burnout"
        ]
    ),

    "Boreout": encontrar_columna(
        df_resultados,
        [
            "kpi_boreout",
            "KPI_Boreout",
            "Boreout"
        ]
    ),

    "Bienestar": encontrar_columna(
        df_resultados,
        [
            "kpi_bienestar",
            "KPI_Bienestar",
            "Bienestar"
        ]
    ),

    "Intención de cambio": encontrar_columna(
        df_resultados,
        [
            "kpi_rotacion",
            "KPI_Rotacion",
            "Rotacion",
            "Intencion_cambio",
            "Intención_cambio"
        ]
    )
}


# =====================================================
# 5. VALIDAR COLUMNAS
# =====================================================

if columna_cultura is None:
    raise KeyError(
        "No se ha encontrado una columna de cultura dominante "
        "o cultura percibida.\n\n"
        "Columnas disponibles:\n"
        + "\n".join(df_resultados.columns.astype(str))
    )


kpis_no_encontrados = [
    kpi
    for kpi, columna in columnas_kpis.items()
    if columna is None
]


if kpis_no_encontrados:
    raise KeyError(
        "No se han encontrado las siguientes columnas KPI: "
        + ", ".join(kpis_no_encontrados)
        + "\n\nColumnas disponibles:\n"
        + "\n".join(df_resultados.columns.astype(str))
    )


# =====================================================
# 6. PREPARAR DATOS
# =====================================================

df_cultura_kpis = pd.DataFrame({
    "Cultura": (
        df_resultados[columna_cultura]
        .apply(normalizar_cultura)
    ),

    "Burnout": pd.to_numeric(
        df_resultados[columnas_kpis["Burnout"]],
        errors="coerce"
    ),

    "Boreout": pd.to_numeric(
        df_resultados[columnas_kpis["Boreout"]],
        errors="coerce"
    ),

    "Bienestar": pd.to_numeric(
        df_resultados[columnas_kpis["Bienestar"]],
        errors="coerce"
    ),

    "Intención de cambio": pd.to_numeric(
        df_resultados[columnas_kpis["Intención de cambio"]],
        errors="coerce"
    )
})


df_cultura_kpis = (
    df_cultura_kpis
    .dropna(subset=["Cultura"])
    .copy()
)


if df_cultura_kpis.empty:
    raise ValueError(
        "No existen datos válidos para analizar "
        "la relación entre cultura y KPIs."
    )


# =====================================================
# 7. MEDIAS POR CULTURA
# =====================================================

resumen_cultura_kpis = (
    df_cultura_kpis
    .groupby(
        "Cultura",
        observed=False
    )[ORDEN_KPIS]
    .mean()
    .reindex(ORDEN_CULTURAS)
    .round(2)
    .reset_index()
)


# Número de participantes por cultura
participantes_por_cultura = (
    df_cultura_kpis["Cultura"]
    .value_counts()
    .reindex(ORDEN_CULTURAS)
    .fillna(0)
    .astype(int)
)


resumen_cultura_kpis["Participantes"] = (
    resumen_cultura_kpis["Cultura"]
    .map(participantes_por_cultura)
)


# Pasar a formato largo para Plotly
resumen_cultura_kpis_largo = (
    resumen_cultura_kpis
    .melt(
        id_vars=[
            "Cultura",
            "Participantes"
        ],
        value_vars=ORDEN_KPIS,
        var_name="KPI",
        value_name="Puntuación media"
    )
)


resumen_cultura_kpis_largo["Cultura"] = pd.Categorical(
    resumen_cultura_kpis_largo["Cultura"],
    categories=ORDEN_CULTURAS,
    ordered=True
)


resumen_cultura_kpis_largo["KPI"] = pd.Categorical(
    resumen_cultura_kpis_largo["KPI"],
    categories=ORDEN_KPIS,
    ordered=True
)


resumen_cultura_kpis_largo = (
    resumen_cultura_kpis_largo
    .sort_values(
        [
            "Cultura",
            "KPI"
        ]
    )
)


# =====================================================
# 8. GRÁFICO
# =====================================================

fig_cultura_kpis = px.bar(
    resumen_cultura_kpis_largo,

    x="Cultura",

    y="Puntuación media",

    color="KPI",

    barmode="group",

    text="Puntuación media",

    color_discrete_map=COLORES_KPIS,

    category_orders={
        "Cultura": ORDEN_CULTURAS,
        "KPI": ORDEN_KPIS
    },

    custom_data=[
        "Participantes"
    ]
)


fig_cultura_kpis.update_traces(
    texttemplate="%{text:.2f}",

    textposition="outside",

    cliponaxis=False,

    hovertemplate=(
        "<b>%{x}</b><br>"
        "KPI: %{fullData.name}<br>"
        "Media: %{y:.2f}<br>"
        "Participantes: %{customdata[0]}"
        "<extra></extra>"
    )
)


fig_cultura_kpis.update_layout(
    template="simple_white",

    title=(
        "Relación entre cultura organizacional "
        "y KPIs de bienestar"
    ),

    width=1100,

    height=620,

    xaxis_title="Cultura organizacional percibida",

    yaxis_title="Puntuación media",

    yaxis=dict(
        range=[0, 5.5],

        dtick=1,

        showgrid=True,

        gridcolor="#ECEFF1",

        zeroline=False
    ),

    legend=dict(
        title="",

        orientation="h",

        yanchor="bottom",

        y=1.02,

        xanchor="center",

        x=0.5
    ),

    font=dict(
        size=13
    ),

    margin=dict(
        l=80,
        r=60,
        t=130,
        b=80
    )
)


# =====================================================
# 9. GUARDAR GRÁFICO
# =====================================================

ruta_grafico_cultura_kpis = (
    CARPETA_VISUALIZACIONES
    / "06_relacion_cultura_kpis.png"
)


fig_cultura_kpis.write_image(
    ruta_grafico_cultura_kpis,
    width=1100,
    height=620,
    scale=3
)




fig_cultura_kpis.show()


# =====================================================
# 10. TABLA RESUMEN
# =====================================================

display(
    resumen_cultura_kpis[
        [
            "Cultura",
            "Participantes",
            "Burnout",
            "Boreout",
            "Bienestar",
            "Intención de cambio"
        ]
    ]
)

,Cultura,Participantes,Burnout,Boreout,Bienestar,Intención de cambio
0,🔵 Adhocracia,6,3.42,2.71,3.25,3.78
1,🟢 Clan,4,3.06,2.81,3.56,3.92
2,🟠 Mercado,12,3.69,3.54,3.04,3.89
3,🟡 Jerárquica,0,NaN,NaN,NaN,NaN


In [18]:
# =====================================================
# SMALL MULTIPLES DE BARRAS
# KPIs SEGÚN VARIABLES DEMOGRÁFICAS Y LABORALES
# =====================================================

import pandas as pd
import plotly.graph_objects as go

from pathlib import Path
from plotly.subplots import make_subplots
from estilos import COLORES_EBLET, aplicar_estilo


# =====================================================
# 1. CONFIGURAR CARPETA DE VISUALIZACIONES
# =====================================================


if "RAIZ_PROYECTO" not in globals():
    raise NameError(
        "No existe la variable 'RAIZ_PROYECTO'. "
        "Ejecuta primero la celda de configuración inicial."
    )


CARPETA_VISUALIZACIONES = (
    Path(RAIZ_PROYECTO)
    / "visualizaciones"
)

CARPETA_VISUALIZACIONES.mkdir(
    parents=True,
    exist_ok=True
)




# =====================================================
# 2. COMPROBAR DATAFRAMES NECESARIOS
# =====================================================

if "df_raw" not in globals():
    raise NameError(
        "No existe el dataframe 'df_raw'. "
        "Ejecuta primero la celda de carga del CSV."
    )


if "df_resultados" not in globals():
    raise NameError(
        "No existe el dataframe 'df_resultados'. "
        "Ejecuta primero la celda de cálculo de KPIs."
    )


# =====================================================
# 3. PREPARAR DATOS DEMOGRÁFICOS
# =====================================================

df_demo = df_raw.copy()


# Limpiar espacios en los nombres de las columnas
df_demo.columns = (
    df_demo.columns
    .astype(str)
    .str.strip()
)


# -----------------------------------------------------
# Detectar automáticamente columnas
# -----------------------------------------------------

def buscar_columna(df, textos_posibles):
    """
    Busca una columna mediante coincidencia exacta
    o parcial con los textos proporcionados.
    """

    columnas = df.columns.tolist()

    # Coincidencias exactas
    for texto in textos_posibles:
        if texto in columnas:
            return texto

    # Coincidencias parciales
    for columna in columnas:

        columna_normalizada = (
            str(columna)
            .strip()
            .lower()
        )

        for texto in textos_posibles:

            texto_normalizado = (
                str(texto)
                .strip()
                .lower()
            )

            if texto_normalizado in columna_normalizada:
                return columna

    return None


columna_alias = buscar_columna(
    df_demo,
    [
        "🎭 Mi alias es",
        "Mi alias es",
        "Alias"
    ]
)


columna_sexo = buscar_columna(
    df_demo,
    [
        "Sexo / Género",
        "Sexo/Género",
        "Sexo",
        "Género"
    ]
)


columna_edad = buscar_columna(
    df_demo,
    [
        "¿ Cuál es tu edad?",
        "¿Cuál es tu edad?",
        "Cuál es tu edad",
        "Edad"
    ]
)


columna_salario = buscar_columna(
    df_demo,
    [
        (
            "¿Cuál es o fue aproximadamente tu salario "
            "bruto anual en esa empresa?"
        ),
        "salario bruto anual",
        "Salario"
    ]
)


columna_jornada = buscar_columna(
    df_demo,
    [
        "¿Cuál es o fue tu jornada laboral?",
        "jornada laboral",
        "Jornada"
    ]
)


columna_modalidad = buscar_columna(
    df_demo,
    [
        "Modalidad de trabajo",
        "modalidad",
        "Modalidad"
    ]
)


print("\nColumnas demográficas detectadas:")
print(f"  Alias:     {columna_alias}")
print(f"  Sexo:      {columna_sexo}")
print(f"  Edad:      {columna_edad}")
print(f"  Salario:   {columna_salario}")
print(f"  Jornada:   {columna_jornada}")
print(f"  Modalidad: {columna_modalidad}")


if columna_alias is None:
    raise KeyError(
        "No se ha encontrado la columna del alias "
        "en df_raw."
    )


# -----------------------------------------------------
# Renombrar columnas
# -----------------------------------------------------

mapa_renombrado = {
    columna_alias: "Alias"
}


if columna_sexo is not None:
    mapa_renombrado[columna_sexo] = "Sexo"


if columna_edad is not None:
    mapa_renombrado[columna_edad] = "Edad"


if columna_salario is not None:
    mapa_renombrado[columna_salario] = "Salario"


if columna_jornada is not None:
    mapa_renombrado[columna_jornada] = "Jornada"


if columna_modalidad is not None:
    mapa_renombrado[columna_modalidad] = "Modalidad"


df_demo = df_demo.rename(
    columns=mapa_renombrado
)


# =====================================================
# 4. COMPROBAR Y LIMPIAR ALIAS
# =====================================================

if "Alias" not in df_resultados.columns:
    raise KeyError(
        "df_resultados no contiene la columna 'Alias'."
    )


df_demo["Alias"] = (
    df_demo["Alias"]
    .astype("string")
    .str.strip()
)


# Trabajar con una copia para no modificar
# accidentalmente el dataframe original.
df_resultados_demografia = df_resultados.copy()


df_resultados_demografia["Alias"] = (
    df_resultados_demografia["Alias"]
    .astype("string")
    .str.strip()
)


# Eliminar alias vacíos
df_demo = df_demo[
    df_demo["Alias"].notna()
    & (df_demo["Alias"] != "")
].copy()


# Conservar una fila por participante
df_demo = df_demo.drop_duplicates(
    subset="Alias",
    keep="first"
)


# =====================================================
# 5. SELECCIONAR VARIABLES DEMOGRÁFICAS
# =====================================================

columnas_demograficas = [
    columna
    for columna in [
        "Alias",
        "Sexo",
        "Edad",
        "Salario",
        "Jornada",
        "Modalidad"
    ]
    if columna in df_demo.columns
]


df_demograficos = df_demo[
    columnas_demograficas
].copy()


# =====================================================
# 6. UNIR KPIs Y DATOS DEMOGRÁFICOS
# =====================================================

columnas_repetidas = [
    columna
    for columna in [
        "Sexo",
        "Edad",
        "Salario",
        "Jornada",
        "Modalidad"
    ]
    if columna in df_resultados_demografia.columns
]


df_resultados_base = (
    df_resultados_demografia
    .drop(
        columns=columnas_repetidas,
        errors="ignore"
    )
)


df_final = df_resultados_base.merge(
    df_demograficos,
    on="Alias",
    how="left",
    validate="one_to_one"
)





# =====================================================
# 7. CREAR GRUPOS DE EDAD
# =====================================================

if "Edad" in df_final.columns:

    df_final["Edad_numerica"] = pd.to_numeric(
        df_final["Edad"],
        errors="coerce"
    )

    if df_final["Edad_numerica"].notna().any():

        df_final["Grupo_Edad"] = pd.cut(
            df_final["Edad_numerica"],
            bins=[
                17,
                30,
                40,
                50,
                float("inf")
            ],
            labels=[
                "18-30 años",
                "31-40 años",
                "41-50 años",
                "Más de 50 años"
            ],
            include_lowest=True
        )

    else:

        # Edad ya expresada mediante intervalos
        df_final["Grupo_Edad"] = (
            df_final["Edad"]
            .astype("string")
            .str.strip()
        )


# =====================================================
# 8. LIMPIAR VARIABLES CATEGÓRICAS
# =====================================================

columnas_categoricas = [
    "Sexo",
    "Grupo_Edad",
    "Salario",
    "Jornada",
    "Modalidad"
]


for columna in columnas_categoricas:

    if columna in df_final.columns:

        df_final[columna] = (
            df_final[columna]
            .astype("string")
            .str.strip()
            .replace({
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "<NA>": pd.NA
            })
        )


# =====================================================
# 9. ORDEN DE LAS CATEGORÍAS
# =====================================================

ordenes_preferidos = {

    "Sexo": [
        "Mujer",
        "Hombre",
        "No binario",
        "Prefiero no decirlo"
    ],

    "Grupo_Edad": [
        "18-30 años",
        "31-40 años",
        "41-50 años",
        "Más de 50 años"
    ],

    "Salario": [
        "Menos de 15.000 €",
        "15.000 € – 25.000 €",
        "25.001 € – 35.000 €",
        "35.001 € – 45.000 €",
        "Más de 45.000 €"
    ],

    "Jornada": [
        "Completa",
        "Parcial"
    ],

    "Modalidad": [
        "Presencial",
        "Híbrido",
        "Remoto"
    ]
}


def obtener_orden_real(
    serie,
    orden_preferido
):
    """
    Conserva el orden predefinido y añade al final
    las categorías adicionales presentes en los datos.
    """

    valores_reales = (
        serie
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    categorias_ordenadas = [
        categoria
        for categoria in orden_preferido
        if categoria in valores_reales
    ]

    categorias_adicionales = [
        categoria
        for categoria in valores_reales
        if categoria not in categorias_ordenadas
    ]

    return (
        categorias_ordenadas
        + categorias_adicionales
    )


for columna, orden_preferido in ordenes_preferidos.items():

    if columna in df_final.columns:

        orden_real = obtener_orden_real(
            df_final[columna],
            orden_preferido
        )

        df_final[columna] = pd.Categorical(
            df_final[columna],
            categories=orden_real,
            ordered=True
        )


# =====================================================
# 10. VARIABLES DEMOGRÁFICAS Y LABORALES
# =====================================================

variables_candidatas = [
    ("Sexo", "Sexo"),
    ("Grupo_Edad", "Edad"),
    ("Salario", "Salario"),
    ("Jornada", "Jornada"),
    ("Modalidad", "Modalidad")
]


variables = [
    (variable, titulo)
    for variable, titulo in variables_candidatas
    if variable in df_final.columns
    and df_final[variable].notna().any()
]


if not variables:
    raise ValueError(
        "No hay variables demográficas válidas "
        "para construir el gráfico."
    )




# =====================================================
# 11. CONFIGURAR KPIs
# =====================================================

kpis = [
    "Burnout",
    "Boreout",
    "Bienestar",
    "Intencion_cambio"
]


columnas_kpi_faltantes = [
    kpi
    for kpi in kpis
    if kpi not in df_final.columns
]


if columnas_kpi_faltantes:
    raise KeyError(
        "Faltan los siguientes KPIs "
        "en df_resultados: "
        + ", ".join(columnas_kpi_faltantes)
    )


etiquetas_kpi = {
    "Burnout": "Burnout",
    "Boreout": "Boreout",
    "Bienestar": "Bienestar",
    "Intencion_cambio": "Intención de cambio"
}


COLORES_KPI_GRAFICO = {
    "Burnout": COLORES_EBLET["Burnout"],
    "Boreout": COLORES_EBLET["Boreout"],
    "Bienestar": COLORES_EBLET["Bienestar"],
    "Intencion_cambio": COLORES_EBLET["Rotacion"]
}


# =====================================================
# 12. CREAR LIENZO
# =====================================================

fig_demograficos = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[
        titulo
        for _, titulo in variables
    ],
    horizontal_spacing=0.08,
    vertical_spacing=0.23
)


posiciones = [
    (1, 1),
    (1, 2),
    (1, 3),
    (2, 1),
    (2, 2)
]


# =====================================================
# 13. CALCULAR RESÚMENES Y AÑADIR GRÁFICOS
# =====================================================

tablas_resumen = []


for indice, (
    (variable, titulo),
    (fila, columna)
) in enumerate(
    zip(
        variables,
        posiciones
    )
):

    # Medias de KPIs por categoría
    resumen = (
        df_final
        .dropna(
            subset=[variable]
        )
        .groupby(
            variable,
            observed=True
        )[kpis]
        .mean()
        .round(2)
        .reset_index()
    )


    # Número de participantes por categoría
    conteos = (
        df_final
        .dropna(
            subset=[variable]
        )
        .groupby(
            variable,
            observed=True
        )
        .size()
        .rename("Participantes")
        .reset_index()
    )


    resumen = resumen.merge(
        conteos,
        on=variable,
        how="left"
    )


    resumen = resumen.dropna(
        subset=kpis,
        how="all"
    )


    # -------------------------------------------------
    # Añadir barras
    # -------------------------------------------------

    for kpi in kpis:

        fig_demograficos.add_trace(
            go.Bar(
                x=(
                    resumen[variable]
                    .astype(str)
                ),

                y=resumen[kpi],

                name=etiquetas_kpi[kpi],

                marker_color=(
                    COLORES_KPI_GRAFICO[kpi]
                ),

                text=resumen[kpi],

                texttemplate="%{text:.2f}",

                textposition="outside",

                cliponaxis=False,

                customdata=resumen[
                    ["Participantes"]
                ],

                showlegend=(indice == 0),

                hovertemplate=(
                    f"<b>{etiquetas_kpi[kpi]}</b><br>"
                    f"{titulo}: %{{x}}<br>"
                    "Media: %{y:.2f}<br>"
                    "Participantes: %{customdata[0]}"
                    "<extra></extra>"
                )
            ),
            row=fila,
            col=columna
        )


    fig_demograficos.update_yaxes(
        range=[0, 5.55],
        dtick=1,
        title_text=(
            "Puntuación media"
            if columna == 1
            else None
        ),
        showgrid=True,
        gridcolor="#ECEFF1",
        zeroline=False,
        row=fila,
        col=columna
    )


    fig_demograficos.update_xaxes(
        title_text=None,
        tickangle=-25,
        automargin=True,
        row=fila,
        col=columna
    )


    # -------------------------------------------------
    # Preparar tabla resumen
    # -------------------------------------------------

    tabla_variable = resumen.rename(
        columns={
            variable: "Categoría",
            "Intencion_cambio": (
                "Intención de cambio"
            )
        }
    ).copy()


    tabla_variable.insert(
        0,
        "Variable",
        titulo
    )


    tablas_resumen.append(
        tabla_variable
    )


# =====================================================
# 14. OCULTAR ESPACIOS VACÍOS
# =====================================================

posiciones_utilizadas = posiciones[
    :len(variables)
]


posiciones_vacias = [
    posicion
    for posicion in posiciones
    if posicion not in posiciones_utilizadas
]


# El sexto panel siempre queda vacío
posiciones_vacias.append(
    (2, 3)
)


for fila, columna in set(posiciones_vacias):

    fig_demograficos.update_xaxes(
        visible=False,
        row=fila,
        col=columna
    )

    fig_demograficos.update_yaxes(
        visible=False,
        row=fila,
        col=columna
    )


# =====================================================
# 15. CONFIGURACIÓN GENERAL DEL GRÁFICO
# =====================================================

fig_demograficos.update_layout(
    title=(
        "<b>KPIs medios según características "
        "demográficas y laborales</b>"
    ),

    barmode="group",

    bargap=0.22,

    bargroupgap=0.05
)


fig_demograficos = aplicar_estilo(
    fig_demograficos,
    ancho=1550,
    alto=1050
)


fig_demograficos.update_layout(
    legend=dict(
        title="",
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.07,
        yanchor="bottom"
    ),

    margin=dict(
        t=180,
        l=80,
        r=50,
        b=145
    )
)


# Estilo de títulos de cada panel
for anotacion in fig_demograficos.layout.annotations:

    anotacion.update(
        font=dict(
            size=16,
            color="#263645"
        )
    )


# =====================================================
# 16. GUARDAR GRÁFICO EN PNG
# =====================================================

ruta_grafico_demograficos = (
    CARPETA_VISUALIZACIONES
    / "03_kpis_segun_variables_demograficas.png"
)


fig_demograficos.write_image(
    ruta_grafico_demograficos,
    width=1550,
    height=1050,
    scale=3
)




fig_demograficos.show()


# =====================================================
# 17. CREAR Y MOSTRAR TABLA RESUMEN
# =====================================================

tabla_resumen_demografica = pd.concat(
    tablas_resumen,
    ignore_index=True
)


columnas_tabla = [
    "Variable",
    "Categoría",
    "Participantes",
    "Burnout",
    "Boreout",
    "Bienestar",
    "Intención de cambio"
]


display(
    tabla_resumen_demografica[
        columnas_tabla
    ]
)


Columnas demográficas detectadas:
  Alias:     🎭 Mi alias es
  Sexo:      Sexo / Género
  Edad:      ¿ Cuál es tu edad?
  Salario:   ¿Cuál es o fue aproximadamente tu salario bruto anual en esa empresa?
  Jornada:   ¿Cuál es o fue tu jornada laboral?
  Modalidad: Modalidad de trabajo


,Variable,Categoría,Participantes,Burnout,Boreout,Bienestar,Intención de cambio
0,Sexo,Mujer,7,3.79,3.64,2.82,3.81
1,Sexo,Hombre,13,3.19,2.85,3.48,3.72
2,Sexo,No binario,2,4.50,3.75,2.62,5.00
3,Edad,18-30 años,1,2.50,3.50,4.00,3.00
4,Edad,31-40 años,9,4.19,3.47,2.86,4.37
5,Edad,41-50 años,10,2.95,2.88,3.42,3.40
6,Edad,Más de 50 años,2,3.62,3.25,3.12,4.34
7,Salario,Menos de 15.000 €,5,3.45,2.55,3.15,4.00
8,Salario,15.000 € – 25.000 €,9,4.06,3.75,3.03,4.22
9,Salario,25.001 € – 35.000 €,7,2.93,2.86,3.43,3.19


In [14]:
# =====================================================
# 8. ESTADÍSTICOS DESCRIPTIVOS
# =====================================================

resumen_kpis = (
    df_resultados[
        [
            "Burnout",
            "Boreout",
            "Bienestar",
            "Intencion_cambio"
        ]
    ]
    .agg(["mean", "median", "std", "min", "max"])
    .T
    .reset_index()
)


resumen_kpis.columns = [
    "Indicador",
    "Media",
    "Mediana",
    "Desviacion_estandar",
    "Minimo",
    "Maximo"
]


resumen_kpis["Indicador"] = resumen_kpis["Indicador"].replace({
    "Intencion_cambio": "Intención de cambio"
})


display(
    resumen_kpis.style.format({
        "Media": "{:.2f}",
        "Mediana": "{:.2f}",
        "Desviacion_estandar": "{:.2f}",
        "Minimo": "{:.2f}",
        "Maximo": "{:.2f}"
    })
)

,Indicador,Media,Mediana,Desviacion_estandar,Minimo,Maximo
0,Burnout,3.50,3.62,1.04,1.50,4.75
1,Boreout,3.18,3.25,0.94,1.00,5.00
2,Bienestar,3.19,3.38,0.85,1.50,4.75
3,Intención de cambio,3.86,4.00,1.03,1.00,5.00


La comparación con el benchmark tiene una finalidad contextual y descriptiva.  
Las diferencias observadas no deben interpretarse como estadísticamente significativas, ya que la muestra es reducida y no probabilística.

Un percentil elevado indica que la puntuación de la persona es superior a la de una parte amplia del benchmark. Su interpretación depende del indicador:

- En burnout, boreout e intención de cambio, un percentil elevado representa mayor exposición al indicador.
- En bienestar, un percentil elevado representa una posición comparativamente más favorable.

In [19]:
# =====================================================
# 16. SÍNTESIS DESCRIPTIVA
# =====================================================

perfil_frecuente = (
    df_resultados["Perfil"]
    .value_counts()
    .index[0]
)

n_perfil_frecuente = (
    df_resultados["Perfil"]
    .value_counts()
    .iloc[0]
)

porcentaje_perfil = (
    n_perfil_frecuente
    / len(df_resultados)
    * 100
)


cultura_frecuente = (
    df_resultados["Cultura_dominante"]
    .value_counts()
    .index[0]
)

n_cultura_frecuente = (
    df_resultados["Cultura_dominante"]
    .value_counts()
    .iloc[0]
)

porcentaje_cultura = (
    n_cultura_frecuente
    / len(df_resultados)
    * 100
)


indicador_medias = {
    "Burnout": df_resultados["Burnout"].mean(),
    "Boreout": df_resultados["Boreout"].mean(),
    "Bienestar": df_resultados["Bienestar"].mean(),
    "Intención de cambio": (
        df_resultados["Intencion_cambio"].mean()
    )
}


indicador_mayor = max(
    indicador_medias,
    key=indicador_medias.get
)


print("\n" + "=" * 75)
print("RESUMEN DESCRIPTIVO DE EBLET-LITE")
print("=" * 75)

print(f"\nParticipantes analizados: {len(df_resultados)}")
print(f"Respuestas descartadas: {len(df_descartadas)}")

print(
    f"\nPerfil más frecuente: {perfil_frecuente} "
    f"({n_perfil_frecuente} participantes; "
    f"{porcentaje_perfil:.1f} %)"
)

print(
    f"Cultura dominante más frecuente: {cultura_frecuente} "
    f"({n_cultura_frecuente} participantes; "
    f"{porcentaje_cultura:.1f} %)"
)

print("\nPuntuaciones medias:")

for indicador, media in indicador_medias.items():
    print(f" - {indicador}: {media:.2f}")

print(
    f"\nIndicador con mayor puntuación media: "
    f"{indicador_mayor} "
    f"({indicador_medias[indicador_mayor]:.2f})"
)

print("\n" + "=" * 75)


RESUMEN DESCRIPTIVO DE EBLET-LITE

Participantes analizados: 22
Respuestas descartadas: 0

Perfil más frecuente: 🟡 Estable Neutro (10 participantes; 45.5 %)
Cultura dominante más frecuente: Mercado (12 participantes; 54.5 %)

Puntuaciones medias:
 - Burnout: 3.50
 - Boreout: 3.18
 - Bienestar: 3.19
 - Intención de cambio: 3.86

Indicador con mayor puntuación media: Intención de cambio (3.86)



In [6]:
df_resultados

,Alias,Perfil,Burnout,Boreout,Bienestar,Rotacion,Cultura_Dominante,Perfil_EBLET,percentil_burnout,percentil_boreout,percentil_bienestar,percentil_rotacion
0,Germán,🟢 Flourishing,1.50,1.00,4.00,3.67,Adhocracia,🟢 Flourishing,61,73,71,11
1,arcano,🟡 Estable Neutro,2.50,3.50,4.00,4.33,Mercado,🟡 Estable Neutro,24,69,60,93
2,zyx_zyx_zyx,🟡 Estable Neutro,3.25,3.00,2.75,3.33,Mercado,🟡 Estable Neutro,81,30,64,69
3,hhdata,🔴 Crítico Dual,4.75,4.75,3.50,4.67,Mercado,🔴 Crítico Dual,70,42,73,80
4,gato21,🔴 Crítico Dual,4.75,3.75,1.50,3.33,Mercado,🔴 Crítico Dual,30,85,12,53
5,Phan,⚫ Desvinculado,4.50,3.25,2.25,5.00,Clan,⚫ Desvinculado,92,67,60,17
6,Wombat,🟠 Quemado Activo,4.00,2.50,2.00,4.67,Adhocracia,🟠 Quemado Activo,84,31,16,56
7,JC,🟡 Estable Neutro,2.75,2.50,4.00,3.67,Clan,🟡 Estable Neutro,84,58,30,44
8,Queen,🔴 Crítico Dual,4.50,3.50,2.00,5.00,Mercado,🔴 Crítico Dual,33,68,82,87
9,mani_0098,🟠 Quemado Activo,3.50,2.50,3.50,2.67,Mercado,🟠 Quemado Activo,12,51,48,90


In [1]:

# IMPORTS Y CONFIGURACIÓN


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import os
import sys

RAIZ_PROYECTO = r"C:\Users\torre\OneDrive\Escritorio\EBLET-People-Analytics\Python"
sys.path.insert(0, os.path.join(RAIZ_PROYECTO, "src"))

# Paleta de colores para perfiles
color_perfil = {
    "🟢 Flourishing": "#2ecc71",
    "🟡 Estable Neutro": "#f1c40f",
    "🟠 Quemado Activo": "#e67e22",
    "🔵 Aburrido Crónico": "#3498db",
    " Crítico Dual": "#e74c3c",
    "⚫ Desvinculado": "#34495e"
}

print("✅ Notebook EBLET-Lite cargado correctamente")
print("📋 Versión: 23 preguntas (cribado individual)")
print("️  Nota: Esta es una versión abreviada de EBLET Enterprise (72 preguntas)")

✅ Notebook EBLET-Lite cargado correctamente
📋 Versión: 23 preguntas (cribado individual)
️  Nota: Esta es una versión abreviada de EBLET Enterprise (72 preguntas)


In [5]:

# CLASIFICADOR DE PERFILES PARA EBLET-LITE


def clasificar_individuo_lite(kpis):
    """
    Clasifica a una persona en uno de los 6 perfiles de bienestar.
    
    Nota: Esta clasificación se basa en indicadores de screening,
    no en medición completa de burnout (MBI-GS).
    """
    
    # Benchmark de referencia (12,500 empleados)
    BENCHMARK = {
        "burnout": {"p25": 2.1, "p50": 2.8, "p75": 3.6},
        "boreout": {"p25": 1.9, "p50": 2.5, "p75": 3.3},
        "bienestar": {"p25": 2.4, "p50": 3.1, "p75": 3.8}
    }
    
    def calcular_percentil(valor, kpi):
        bench = BENCHMARK[kpi]
        if valor <= bench["p25"]:
            return int((valor / bench["p25"]) * 25)
        elif valor <= bench["p50"]:
            return 25 + int(((valor - bench["p25"]) / (bench["p50"] - bench["p25"])) * 25)
        elif valor <= bench["p75"]:
            return 50 + int(((valor - bench["p50"]) / (bench["p75"] - bench["p50"])) * 25)
        else:
            return 75 + int(((valor - bench["p75"]) / (5 - bench["p75"])) * 25)
    
    percentiles = {
        "burnout": calcular_percentil(kpis["burnout"], "burnout"),
        "boreout": calcular_percentil(kpis["boreout"], "boreout"),
        "bienestar": calcular_percentil(kpis["bienestar"], "bienestar")
    }
    
    # Clasificación basada en umbrales
    if kpis["burnout"] < 2.5 and kpis["boreout"] < 2.5 and kpis["bienestar"] >= 3.5:
        perfil = "flourishing"
        nombre = " Flourishing"
        descripcion = "Indicadores de bienestar óptimo. Rendimiento y equilibrio adecuados."
    elif kpis["burnout"] >= 3.5 and kpis["boreout"] >= 3.5:
        perfil = "critico"
        nombre = " Crítico Dual"
        descripcion = "Indicadores elevados de agotamiento y desconexión. Requiere atención prioritaria."
    elif kpis["burnout"] >= 3.5:
        perfil = "quemado"
        nombre = "🟠 Quemado Activo"
        descripcion = "Indicadores de agotamiento elevados. Carga de trabajo y falta de recuperación."
    elif kpis["boreout"] >= 3.5:
        perfil = "aburrido"
        nombre = "🔵 Aburrido Crónico"
        descripcion = "Indicadores de desconexión elevados. Falta de estimulación y retos."
    elif kpis["rotacion"] >= 3.5 and kpis["bienestar"] < 3.0:
        perfil = "vuelo"
        nombre = "⚫ Desvinculado"
        descripcion = "Intención de cambio laboral elevada con bienestar bajo."
    else:
        perfil = "estable"
        nombre = "🟡 Estable Neutro"
        descripcion = "Indicadores en zona intermedia. Espacio de mejora."
    
    return {
        "perfil": perfil,
        "nombre": nombre,
        "descripcion": descripcion,
        "kpis": kpis,
        "percentiles": percentiles
    }

print("✅ Función clasificar_individuo_lite definida")


✅ Función clasificar_individuo_lite definida


In [6]:

# PROCESAR RESPUESTAS REALES


print("🔧 Procesando respuestas reales del CSV...")

# Encontrar columna de alias
col_alias = encontrar_columna(df_raw, 'alias')

if col_alias is None:
    print("⚠️  No se encontró la columna de alias. Usando 'Respondiente_1', etc.")
    aliases = [f"Respondiente_{i+1}" for i in range(len(df_raw))]
else:
    aliases = df_raw[col_alias].fillna('Anónimo').tolist()

# Construir DataFrame de respuestas mapeadas
resultados_reales = []

for idx, row in df_raw.iterrows():
    resp = {'alias': aliases[idx]}
    
    for q_num, texto_busqueda in mapeo_preguntas.items():
        col_encontrada = encontrar_columna(df_raw, texto_busqueda)
        
        if col_encontrada:
            valor = row[col_encontrada]
            try:
                valor_num = int(valor)
                if 1 <= valor_num <= 5:
                    resp[q_num] = valor_num
                else:
                    resp[q_num] = 3
            except:
                resp[q_num] = 3
        else:
            resp[q_num] = 3
    
    # Calcular KPIs y clasificar
    kpis = calcular_kpis_lite(resp)
    perfil = clasificar_individuo_lite(kpis)
    
    resultados_reales.append({
        'alias': aliases[idx],
        'perfil': perfil['perfil'],
        'perfil_nombre': perfil['nombre'],
        'burnout': round(kpis['burnout'], 2),
        'boreout': round(kpis['boreout'], 2),
        'bienestar': round(kpis['bienestar'], 2),
        'rotacion': round(kpis['rotacion'], 2),
        'cultura_dominante': kpis.get('cultura_dominante', 'N/A'),
        'percentil_burnout': perfil['percentiles'].get('burnout', 50),
        'percentil_boreout': perfil['percentiles'].get('boreout', 50),
        'percentil_bienestar': perfil['percentiles'].get('bienestar', 50),
        '_kpis': kpis,
        '_perfil': perfil
    })

df_reales = pd.DataFrame(resultados_reales)

print(f"\n✅ {len(df_reales)} respuestas procesadas correctamente")

# Mostrar resumen

print("📊 RESUMEN DE PARTICIPANTES")
print(df_reales[['alias', 'perfil_nombre', 'cultura_dominante', 'burnout', 'boreout', 'bienestar']].to_string(index=False))

🔧 Procesando respuestas reales del CSV...

✅ 16 respuestas procesadas correctamente
📊 RESUMEN DE PARTICIPANTES
      alias      perfil_nombre cultura_dominante  burnout  boreout  bienestar
     Germán        Flourishing        Adhocracia     1.50     1.00       4.00
     arcano 🔵 Aburrido Crónico           Mercado     2.50     3.50       4.00
zyx_zyx_zyx   🟡 Estable Neutro           Mercado     3.25     3.00       2.75
     hhdata       Crítico Dual           Mercado     4.75     4.75       3.50
     gato21       Crítico Dual           Mercado     4.75     3.75       1.50
       Phan   🟠 Quemado Activo              Clan     4.50     3.25       2.25
     Wombat   🟠 Quemado Activo        Adhocracia     4.00     2.50       2.00
         JC   🟡 Estable Neutro              Clan     2.75     2.50       4.00
      Queen       Crítico Dual           Mercado     4.50     3.50       2.00
  mani_0098   🟠 Quemado Activo           Mercado     3.50     2.50       3.50
         PS 🔵 Aburrido Crónico 